# Titanic survival prediction: logistic regression with NumPy

This notebook develops a six-feature baseline using logistic regression and
full-batch gradient descent implemented directly with NumPy. Pydantic models
hold the learned preprocessing statistics and training results.

**Workflow:** raw labeled rows → local training/validation split → fit preprocessing
on local training → train → evaluate → fit a fresh pipeline on all labeled rows
→ predict Kaggle passengers → save `submission.csv`.

To use on Kaggle:

1. Import this `.ipynb` file into a Python notebook.
2. Add the **Titanic - Machine Learning from Disaster** competition as an input.
3. Run the cells from top to bottom. The final cell writes
   `/kaggle/working/submission.csv`, which you can submit to the competition.

This is self-contained: it contains the model functions and requires no project
files apart from `train.csv` and `test.csv`. It also runs locally when those two
CSVs are beside the notebook. Use Pydantic 2.x. This notebook does not install
packages or submit predictions automatically.

## 1. Imports and settings

Each row represents one passenger. `Survived = 1` means the passenger survived;
`Survived = 0` means the passenger did not survive.

The settings match the command-line project. Change them here to experiment.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, FileLink
from pydantic import BaseModel, ConfigDict, Field
from rich.console import Console

console = Console()

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
continuous_features = ["Age", "Fare"]

seed = 42
validation_fraction = 0.2
settings = {
    "nu": 0.1,
    "max_epochs": 20000,
    "tolerance": 1e-8,
    "patience": 20,
    "log_every": 2000,
}

## 2. Load the competition data

`train.csv` contains known survival labels. Kaggle's `test.csv` contains the
passengers whose labels are hidden. We will use an internal validation split
of `train.csv` to measure accuracy.

The next cell checks common Kaggle input locations, then the current folder.
Set `DATA_DIR` to a folder yourself if your attached input uses another path.

In [2]:
DATA_DIR = None  # Example override: Path("/kaggle/input/titanic")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()

if DATA_DIR is None:
    candidates = [
        Path("/kaggle/input/competitions/titanic"),
        Path("/kaggle/input/titanic"),
        Path.cwd(),
    ]
    DATA_DIR = next(
        (folder for folder in candidates
         if (folder / "train.csv").is_file() and (folder / "test.csv").is_file()),
        None,
    )

if DATA_DIR is None:
    raise FileNotFoundError(
        "Add the Titanic competition input, or set DATA_DIR to the folder "
        "containing train.csv and test.csv."
    )
DATA_DIR = Path(DATA_DIR)

labeled = pd.read_csv(DATA_DIR / "train.csv")
test_data = pd.read_csv(DATA_DIR / "test.csv")
if test_data.empty or "PassengerId" not in test_data:
    raise ValueError("test.csv must contain passengers and a PassengerId column.")
passenger_ids = test_data["PassengerId"].copy()
if passenger_ids.isna().any() or passenger_ids.duplicated().any():
    raise ValueError("Test PassengerId values must be present and unique.")

print(f"Labeled passengers: {len(labeled)}; Kaggle test passengers: {len(test_data)}")
display(labeled.head())
display(pd.DataFrame({
    "Missing in train.csv": labeled[features].isna().sum(),
    "Missing in test.csv": test_data[features].isna().sum(),
}))

Labeled passengers: 891; Kaggle test passengers: 418


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,Missing in train.csv,Missing in test.csv
Pclass,0,0
Sex,0,0
Age,177,86
SibSp,0,0
Parch,0,0
Fare,0,1


## 3. Define preprocessing

We keep the original feature order and the fixed mapping `male = 0`, `female = 1`.
Only **Age** and **Fare** are median-imputed and standardized. `Pclass` keeps its
original ordinal numeric representation; `SibSp` and `Parch` remain counts.

For each continuous feature, the fitted transformation is

$$x' = \frac{x - \mu_{\mathrm{train}}}{\sigma_{\mathrm{train}}}.$$

The mean and population standard deviation are calculated **after filling
missing values in the training rows**. A constant feature uses a scale of 1.
Defining these functions does not fit any statistics yet.

Pydantic checks that the containers are pandas Series or NumPy arrays. The
preprocessing and training functions perform the numerical checks.

In [3]:
class Preprocessing(BaseModel):
    # Keep pandas Series as Series, with Pydantic checking their container types.
    model_config = ConfigDict(arbitrary_types_allowed=True)

    medians: pd.Series
    means: pd.Series
    scales: pd.Series


class TrainingResult(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True, allow_inf_nan=False)

    W: np.ndarray
    b: float
    epochs: int = Field(ge=1)
    loss: float = Field(ge=0)
    converged: bool

In [4]:
def encode_features(data):
    """Select the same six columns in the same order without changing data."""
    missing = set(features) - set(data.columns)
    if missing:
        raise ValueError(f"Missing feature columns: {sorted(missing)}")

    X = data[features].copy()
    if not X["Sex"].isin(["male", "female"]).all():
        raise ValueError("Sex must contain only 'male' and 'female', with no missing values.")
    X["Sex"] = X["Sex"].map({"male": 0, "female": 1})
    X = X.astype(float)

    if np.isinf(X.to_numpy()).any():
        raise ValueError("Features cannot contain infinity.")
    if X.drop(columns=continuous_features).isna().any().any():
        raise ValueError("Only Age and Fare may contain missing feature values.")
    return X


def fit_preprocessing(training_data):
    """Learn imputation and scaling from the current training rows only."""
    X = encode_features(training_data)
    medians = X[continuous_features].median()
    if medians.isna().any():
        raise ValueError("Training data needs at least one observed Age and Fare.")

    continuous = X[continuous_features].fillna(medians)
    means = continuous.mean()
    # ddof=0 uses the population standard deviation. A constant column stays safe.
    scales = continuous.std(ddof=0).replace(0.0, 1.0)
    return Preprocessing(medians=medians, means=means, scales=scales)


def transform_features(data, preprocessing):
    """Reuse learned statistics; never fit new statistics on validation/test."""
    X = encode_features(data)
    continuous = X[continuous_features].fillna(preprocessing.medians)
    X[continuous_features] = (continuous - preprocessing.means) / preprocessing.scales
    X = X.to_numpy(dtype=float)
    if not np.isfinite(X).all():
        raise ValueError("Preprocessing produced missing or non-finite values.")
    return X

## 4. Split raw labeled rows, then fit local preprocessing

The split is reproducible and stratified: both survival classes appear in each
part. Rounding the held-out count up within each class gives approximately 80/20.
With the supplied dataset this means 712 local training rows and 179 validation
rows.

Fit medians, means, and scales on **local training rows only**. Reuse those exact
values to transform validation rows. The validation labels never enter training.

In [5]:
def split_labeled_data(data, validation_fraction=0.2, seed=42):
    """Make a reproducible split with both survival classes in each part."""
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1.")
    if "Survived" not in data or not data["Survived"].isin([0, 1]).all():
        raise ValueError("train.csv must contain Survived labels equal to 0 or 1.")

    rng = np.random.default_rng(seed)
    train_indices, validation_indices = [], []
    Y = data["Survived"].to_numpy()
    for label in (0, 1):
        indices = rng.permutation(np.flatnonzero(Y == label))
        if len(indices) < 2:
            raise ValueError("Each survival class needs at least two labeled rows.")
        # Round up per class; keep at least one row of each class for training.
        n_validation = min(int(np.ceil(len(indices) * validation_fraction)), len(indices) - 1)
        validation_indices.extend(indices[:n_validation])
        train_indices.extend(indices[n_validation:])

    train = data.iloc[rng.permutation(train_indices)].reset_index(drop=True)
    validation = data.iloc[rng.permutation(validation_indices)].reset_index(drop=True)
    return train, validation

In [6]:
local_train, validation = split_labeled_data(labeled, validation_fraction, seed)
local_preprocessing = fit_preprocessing(local_train)

X_train = transform_features(local_train, local_preprocessing)
X_validation = transform_features(validation, local_preprocessing)
Y_train = local_train["Survived"].to_numpy(dtype=float)
Y_validation = validation["Survived"].to_numpy(dtype=float)

print(f"Local training shape: {X_train.shape}; validation shape: {X_validation.shape}")
display(pd.DataFrame({
    "Training median": local_preprocessing.medians,
    "Training mean after imputation": local_preprocessing.means,
    "Training scale": local_preprocessing.scales,
}))

Local training shape: (712, 6); validation shape: (179, 6)


,Training median,Training mean after imputation,Training scale
Age,28.0000,28.999537,12.755857
Fare,14.4542,32.429617,51.660062


## 5. The manual logistic-regression calculations

For $m$ passengers, $X$ has shape $(m, 6)$, $W$ has shape $(6,)$, and $b$ is a scalar.

$$Z = XW+b, \qquad \hat p = \sigma(Z).$$

The mean binary cross-entropy and gradients are

$$J = -\frac{1}{m}\sum_i\left[Y_i\log(\hat p_i)+(1-Y_i)\log(1-\hat p_i)\right],$$

$$\nabla_W J = \frac{1}{m}X^T(\hat p-Y), \qquad
\frac{\partial J}{\partial b}=\mathrm{mean}(\hat p-Y).$$

`L_cal(Y, Z)` computes the same loss directly from **logits**, using `logaddexp`
for numerical stability. The sigmoid uses separate positive and negative
branches so it never exponentiates a large positive number.

In [7]:
def z_cal(x, w, b):
    return x @ w + b


def sig_cal(z):
    """Compute sigmoid without exponentiating a large positive number."""
    z = np.asarray(z, dtype=float)
    p_hat = np.empty_like(z)
    positive = z >= 0
    p_hat[positive] = 1 / (1 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    p_hat[~positive] = exp_z / (1 + exp_z)
    return p_hat


def L_cal(y, z):
    """Mean binary cross-entropy from logits z, NOT probabilities.

    For binary y this equals mean(log(1 + exp(z)) - y*z).
    The signed-logit form also avoids subtracting nearly equal large values.
    """
    return float(np.mean(np.logaddexp(0, (1 - 2 * y) * z)))


def calculate_gradients(X, Y, p_hat):
    error = p_hat - Y
    dJ_w = X.T @ error / len(Y)
    dJ_b = float(np.mean(error))
    return dJ_w, dJ_b


def training(dJ_w, dJ_b, W, b, nu):
    return W - nu * dJ_w, b - nu * dJ_b

## 6. Train the local model

Each epoch uses all local training passengers for one gradient update:

$$W \leftarrow W-\nu\nabla_WJ, \qquad
b \leftarrow b-\nu\frac{\partial J}{\partial b}.$$

Training starts with zero weights and bias. It stops after 20 consecutive
nonnegative loss improvements smaller than $10^{-8}$, or at the epoch limit.
These defaults come from `settings` above. Logs report the loss **after** an
update, so the final loss describes the returned parameters.

In [8]:
def train_model(
    X, Y, nu=0.1, max_epochs=20000, tolerance=1e-8,
    patience=20, log_every=2000, verbose=True,
):
    """Full-batch gradient descent: one update uses every training passenger."""
    X, Y = np.asarray(X, dtype=float), np.asarray(Y, dtype=float)
    if X.ndim != 2 or Y.ndim != 1 or len(X) != len(Y) or len(Y) == 0:
        raise ValueError("X must be a nonempty matrix and Y one label per row.")
    if not np.isfinite(X).all() or not np.isin(Y, [0, 1]).all():
        raise ValueError("X must be finite and Y must contain binary labels.")
    if not np.isfinite(nu) or nu <= 0 or max_epochs < 1:
        raise ValueError("Learning rate and epoch limit must be positive.")
    if not np.isfinite(tolerance) or tolerance < 0 or patience < 1 or log_every < 1:
        raise ValueError("Tolerance must be nonnegative; patience and log interval positive.")

    W = np.zeros(X.shape[1])
    b = 0.0
    Z = z_cal(X, W, b)
    previous_loss = L_cal(Y, Z)
    small_improvements = 0
    converged = False
    if verbose:
        console.print(f"Epoch {0:>6}: loss={previous_loss:.8f}")

    for epoch in range(1, max_epochs + 1):
        p_hat = sig_cal(Z)
        dJ_w, dJ_b = calculate_gradients(X, Y, p_hat)
        W, b = training(dJ_w, dJ_b, W, b, nu)

        # Calculate loss AFTER the update so it describes the returned W and b.
        Z = z_cal(X, W, b)
        loss = L_cal(Y, Z)
        if not np.isfinite(loss) or not np.isfinite(W).all() or not np.isfinite(b):
            raise ValueError("Training became non-finite. Try a smaller learning rate.")
        if loss > previous_loss + 1e-12:
            raise ValueError("Training loss increased. Try a smaller learning rate.")

        improvement = previous_loss - loss
        if 0 <= improvement < tolerance:
            small_improvements += 1
        else:
            small_improvements = 0
        converged = small_improvements >= patience

        if verbose and (epoch % log_every == 0 or converged or epoch == max_epochs):
            console.print(f"Epoch {epoch:>6}: loss={loss:.8f}, improvement={improvement:.2e}")
        if converged:
            break
        previous_loss = loss

    if verbose:
        reason = "Converged" if converged else "Reached epoch limit"
        console.print(f"{reason} after {epoch} epochs. Final loss={loss:.8f}")
    return TrainingResult(W=W, b=b, epochs=epoch, loss=loss, converged=converged)

In [9]:
local_model = train_model(X_train, Y_train, **settings)

Epoch      0: loss=0.69314718

Epoch   2000: loss=0.44999684, improvement=2.60e-07

Epoch   3776: loss=0.44986245, improvement=9.65e-09

Converged after 3776 epochs. Final loss=0.44986245

## 7. Evaluate the frozen model

Validation follows one forward path: transformed passengers → logits →
probabilities → predicted labels → comparison with known labels. A probability
of at least 0.5 becomes `Survived = 1`.

There are no weight updates or new preprocessing statistics during evaluation.
The confusion matrix uses actual labels as rows and predictions as columns:
`[[true negatives, false positives], [false negatives, true positives]]`.

In [10]:
def predict(X, W, b):
    return (sig_cal(z_cal(X, W, b)) >= 0.5).astype(int)


def evaluate_model(X, Y, W, b):
    """Evaluate frozen parameters; rows are actual, columns are predicted."""
    predictions = predict(X, W, b)
    confusion = np.array([
        [np.sum((Y == 0) & (predictions == 0)), np.sum((Y == 0) & (predictions == 1))],
        [np.sum((Y == 1) & (predictions == 0)), np.sum((Y == 1) & (predictions == 1))],
    ])
    return {
        "accuracy": float(np.mean(predictions == Y)),
        "loss": L_cal(Y, z_cal(X, W, b)),
        "confusion_matrix": confusion,
    }

In [11]:
validation_metrics = evaluate_model(
    X_validation, Y_validation, local_model.W, local_model.b
)
print(f"Validation accuracy: {validation_metrics['accuracy']:.2%}")
print(f"Validation loss: {validation_metrics['loss']:.6f}")
display(pd.DataFrame(
    validation_metrics["confusion_matrix"],
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
))

Validation accuracy: 80.45%
Validation loss: 0.424898


,Predicted 0,Predicted 1
Actual 0,95,15
Actual 1,20,49


## 8. Fit a fresh final pipeline on all labeled passengers

Now fit **new** preprocessing statistics on all of `train.csv` and train a
**new** model from zero. The rows previously held out for local validation can
now contribute to this final fit. Keep the local validation result as the
estimate from the earlier held-out evaluation.

Transform Kaggle's test passengers with the full-training statistics.

In [12]:
final_preprocessing = fit_preprocessing(labeled)
X_all = transform_features(labeled, final_preprocessing)
X_test = transform_features(test_data, final_preprocessing)
Y_all = labeled["Survived"].to_numpy(dtype=float)

print(f"Final training on all {len(labeled)} labeled passengers")
final_model = train_model(X_all, Y_all, **settings)

Final training on all 891 labeled passengers


Epoch      0: loss=0.69314718

Epoch   2000: loss=0.44303650, improvement=3.39e-07

Epoch   3850: loss=0.44286628, improvement=9.63e-09

Converged after 3850 epochs. Final loss=0.44286628

## 9. Create the Kaggle submission

There are no true survival labels in `test.csv`, so this step produces predictions.
The CSV must contain exactly `PassengerId` and `Survived`, in the original
passenger order, with no saved DataFrame index.

On Kaggle the output is `/kaggle/working/submission.csv`. Locally it is written
in the current folder. Saving the CSV does not submit it to the competition.

In [13]:
submission = pd.DataFrame({
    "PassengerId": passenger_ids.to_numpy(),
    "Survived": predict(X_test, final_model.W, final_model.b),
})

assert list(submission.columns) == ["PassengerId", "Survived"]
assert len(submission) == len(test_data)
assert submission["PassengerId"].tolist() == passenger_ids.tolist()
assert submission["Survived"].isin([0, 1]).all()
assert pd.api.types.is_integer_dtype(submission["Survived"])
assert not submission.isna().any().any()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = OUTPUT_DIR / "submission.csv"
if submission_path.resolve() in {
    (DATA_DIR / "train.csv").resolve(), (DATA_DIR / "test.csv").resolve(),
}:
    raise ValueError("The submission path must differ from the input CSV paths.")
submission.to_csv(submission_path, index=False)

print(f"Saved {len(submission)} predictions to {submission_path.name}")
display(submission.head())
# Relative links work when the notebook runs in Kaggle's working folder.
if submission_path.parent.resolve() == Path.cwd().resolve():
    display(FileLink(submission_path.name))

Saved 418 predictions to submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


submission.csv

## Interpreting the result

The displayed accuracy is a **local validation estimate**, not a Kaggle score.
Repeatedly choosing settings against the same validation rows can make that
estimate optimistic. Kaggle scores the submitted predictions against labels
hidden from this notebook.

This baseline deliberately keeps the original six features. Possible later
experiments include different encodings of passenger class or new features,
with each learned transformation fitted only on the appropriate training rows.

References: [Kaggle notebook documentation](https://www.kaggle.com/docs/notebooks),
[NumPy logaddexp](https://numpy.org/doc/stable/reference/generated/numpy.logaddexp.html),
and [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/).